In [1]:
# ==============================
# 1. Imports
# ==============================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, RandomSampler, SequentialSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
import re
import nltk
from nltk.corpus import stopwords
import warnings

warnings.filterwarnings('ignore')
nltk.download('stopwords', quiet=True)

# ✅ Device
torch.backends.cudnn.enabled = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# ==============================
# 2. Config
# ==============================
MAX_LEN    = 128
BATCH_SIZE = 32
EPOCHS     = 10
LR         = 1e-3
HIDDEN_DIM = 128
MODEL_NAME = "microsoft/deberta-v3-base"

# ==============================
# 3. Load & Preprocess Data
# ==============================
df = pd.read_csv("/kaggle/input/datasets/najninsultanashirin/spam-dataset-qtl/spamdata_v2 (1).csv")
df = df.dropna(subset=['text', 'label'])

stop_words = set(stopwords.words('english'))

def clean(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = text.lower()
    words = [w for w in text.split() if w not in stop_words]
    return " ".join(words)

df['text'] = df['text'].apply(clean)

le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])

# Balance classes
majority = df['label'].value_counts().idxmax()
minority = df['label'].value_counts().idxmin()
df_maj = df[df['label'] == majority].sample(len(df[df['label'] == minority]), random_state=42)
df = pd.concat([df_maj, df[df['label'] == minority]]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Balanced dataset: {df['label'].value_counts().to_dict()}")

# ==============================
# 4. Split
# ==============================
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'], df['label'], test_size=0.2, stratify=df['label'], random_state=42
)

# ==============================
# 5. Tokenization
# ==============================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(texts):
    return tokenizer(
        texts.tolist(),
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

train_enc = encode(train_texts)
val_enc   = encode(val_texts)

def make_loader(enc, labels, sampler_cls):
    dataset = TensorDataset(
        enc['input_ids'],
        enc['attention_mask'],
        torch.tensor(labels.values, dtype=torch.long)
    )
    return DataLoader(dataset, sampler=sampler_cls(dataset), batch_size=BATCH_SIZE)

train_loader = make_loader(train_enc, train_labels, RandomSampler)
val_loader   = make_loader(val_enc,   val_labels,   SequentialSampler)

# ==============================
# 6. Metrics Helper
# ==============================
def compute_all_metrics(all_labels, all_preds, all_probs, name):
    # Guard against any remaining NaN
    all_probs = np.nan_to_num(np.array(all_probs), nan=0.5)

    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
    rec  = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
    spec = recall_score(all_labels, all_preds, pos_label=0, average='binary', zero_division=0)
    f1   = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
    auc  = roc_auc_score(all_labels, all_probs)

    print(f"\n{'='*55}")
    print(f"  {name} — Final Results")
    print(f"{'='*55}")
    print(f"  Accuracy    : {acc:.4f}")
    print(f"  Precision   : {prec:.4f}")
    print(f"  Recall      : {rec:.4f}")
    print(f"  Specificity : {spec:.4f}")
    print(f"  F1-score    : {f1:.4f}")
    print(f"  AUC         : {auc:.4f}")
    print(f"\n{classification_report(all_labels, all_preds)}")
    return dict(accuracy=acc, precision=prec, recall=rec,
                specificity=spec, f1=f1, auc=auc)

# ==============================
# 7. Shared Training Loop
# ==============================
def train_and_evaluate(model, train_loader, val_loader, model_name, epochs=EPOCHS, lr=LR):
    model.to(DEVICE)
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer   = AdamW(trainable, lr=lr)
    total_steps = len(train_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=0, num_training_steps=total_steps
    )
    loss_fn = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        # ── Train ──
        model.train()
        t_loss, t_correct, t_total = 0, 0, 0
        for batch in train_loader:
            ids, mask, lbls = [b.to(DEVICE) for b in batch]
            optimizer.zero_grad()
            logits = model(ids, mask)
            loss   = loss_fn(logits, lbls)
            if torch.isnan(loss):
                continue
            loss.backward()
            nn.utils.clip_grad_norm_(trainable, 1.0)
            optimizer.step()
            scheduler.step()
            t_loss    += loss.item()
            t_correct += (logits.argmax(1) == lbls).sum().item()
            t_total   += lbls.size(0)

        # ── Validate ──
        model.eval()
        v_loss, v_correct, v_total = 0, 0, 0
        all_preds, all_labels, all_probs = [], [], []
        with torch.no_grad():
            for batch in val_loader:
                ids, mask, lbls = [b.to(DEVICE) for b in batch]
                logits = model(ids, mask)
                loss   = loss_fn(logits, lbls)
                probs  = F.softmax(logits, dim=1)[:, 1].cpu().float().numpy()
                preds  = logits.argmax(1).cpu().numpy()
                v_loss    += loss.item() if not torch.isnan(loss) else 0
                v_correct += (logits.argmax(1) == lbls).sum().item()
                v_total   += lbls.size(0)
                all_preds.extend(preds)
                all_labels.extend(lbls.cpu().numpy())
                all_probs.extend(probs)

        avg_t = t_loss / max(len(train_loader), 1)
        avg_v = v_loss / max(len(val_loader), 1)
        v_acc = v_correct / max(v_total, 1)
        history['train_loss'].append(avg_t)
        history['val_loss'].append(avg_v)
        history['val_acc'].append(v_acc)

        print(f"[{model_name}] Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {avg_t:.4f} | Val Loss: {avg_v:.4f} | Val Acc: {v_acc:.4f}")

    metrics = compute_all_metrics(
        np.array(all_labels), np.array(all_preds), np.array(all_probs), model_name
    )
    return history, metrics

# ==============================
# 8. DeBERTa-BiLSTM + MLP model
# ==============================
class DebertaBiLSTM_MLP(nn.Module):
    def __init__(self, hidden_dim=128, mlp_hidden=64):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)

        # Freeze DeBERTa encoder
        for param in self.bert.parameters():
            param.requires_grad = False

        self.lstm = nn.LSTM(
            input_size=768,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )
        self.dropout = nn.Dropout(0.3)

        # MLP head
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(mlp_hidden, 2)
        )

    def forward(self, ids, mask):
        with torch.no_grad():
            outputs = self.bert(input_ids=ids, attention_mask=mask)

        x = outputs.last_hidden_state.detach().float()
        lstm_out, _ = self.lstm(x)
        mask_exp = mask.unsqueeze(-1).float()
        pooled = (lstm_out * mask_exp).sum(1) / mask_exp.sum(1).clamp(min=1e-9)
        return self.mlp(self.dropout(pooled))

# ==============================
# 9. Run DeBERTa-BiLSTM + MLP
# ==============================
print("\n" + "="*55)
print("MODEL: DeBERTa-BiLSTM + MLP")
print("="*55)
model = DebertaBiLSTM_MLP(hidden_dim=128, mlp_hidden=64)
history, metrics = train_and_evaluate(
    model, train_loader, val_loader, "DeBERTa-BiLSTM + MLP", lr=1e-3
)

# Optionally print metrics again
print("\nFinal metrics:")
for k, v in metrics.items():
    print(f"{k:<12}: {v:.4f}")

Using device: cuda
Balanced dataset: {1: 747, 0: 747}


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]


MODEL: DeBERTa-BiLSTM + MLP


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

[DeBERTa-BiLSTM + MLP] Epoch 1/10 | Train Loss: 0.4333 | Val Loss: 0.2912 | Val Acc: 0.8829
[DeBERTa-BiLSTM + MLP] Epoch 2/10 | Train Loss: 0.2992 | Val Loss: 0.2986 | Val Acc: 0.8829
[DeBERTa-BiLSTM + MLP] Epoch 3/10 | Train Loss: 0.2975 | Val Loss: 0.2877 | Val Acc: 0.8930
[DeBERTa-BiLSTM + MLP] Epoch 4/10 | Train Loss: 0.3009 | Val Loss: 0.2756 | Val Acc: 0.8696
[DeBERTa-BiLSTM + MLP] Epoch 5/10 | Train Loss: 0.2620 | Val Loss: 0.2817 | Val Acc: 0.8829
[DeBERTa-BiLSTM + MLP] Epoch 6/10 | Train Loss: 0.2415 | Val Loss: 0.2403 | Val Acc: 0.9231
[DeBERTa-BiLSTM + MLP] Epoch 7/10 | Train Loss: 0.2090 | Val Loss: 0.2338 | Val Acc: 0.9030
[DeBERTa-BiLSTM + MLP] Epoch 8/10 | Train Loss: 0.2127 | Val Loss: 0.2030 | Val Acc: 0.9264
[DeBERTa-BiLSTM + MLP] Epoch 9/10 | Train Loss: 0.2040 | Val Loss: 0.2070 | Val Acc: 0.9164
[DeBERTa-BiLSTM + MLP] Epoch 10/10 | Train Loss: 0.1860 | Val Loss: 0.2019 | Val Acc: 0.9231

  DeBERTa-BiLSTM + MLP — Final Results
  Accuracy    : 0.9231
  Precision   : 

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder
from transformers import AutoModel, AutoTokenizer
from datasets import load_dataset
import nltk
from nltk.corpus import stopwords
import re
import warnings
from torch.utils.data import DataLoader, TensorDataset
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

warnings.filterwarnings('ignore')
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# ─────────────────────────────────────────────
# TEXT PREPROCESSING
# ─────────────────────────────────────────────
def preprocess_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text.lower()

def clean_text(text):
    text = preprocess_text(str(text))
    text = re.sub(r'\d+', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

# ─────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────
def load_mpqa():
    print("\nLoading MPQA dataset...")
    ds = load_dataset("jxm/mpqa")
    df = ds["train"].to_pandas()

    df["text"] = df["sentence"].apply(clean_text)
    df = df[df["text"].str.strip() != ""]

    le = LabelEncoder()
    df["label"] = le.fit_transform(df["label"])

    # Balance dataset
    min_class = df["label"].value_counts().min()
    df = df.groupby("label").sample(min_class, random_state=42)

    print(df["label"].value_counts())

    return df["text"].tolist(), df["label"].tolist(), le

# ─────────────────────────────────────────────
# MODEL (FIXED)
# ─────────────────────────────────────────────
class DeBERTaBiLSTM(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # ✅ FORCE FLOAT32 (fix)
        self.bert = AutoModel.from_pretrained(
            "microsoft/deberta-v3-base",
            torch_dtype=torch.float32
        )

        self.lstm = nn.LSTM(
            input_size=768,
            hidden_size=128,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)

        # ✅ CRITICAL FIX
        x = outputs.last_hidden_state.float()

        x, _ = self.lstm(x)
        x = x[:, -1, :]
        return self.fc(x)

# ─────────────────────────────────────────────
# TOKENIZATION + SPLIT (TRAIN / VAL / TEST)
# ─────────────────────────────────────────────
def prepare_data(texts, labels, tokenizer):
    enc = tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")

    X_ids, X_mask = enc["input_ids"], enc["attention_mask"]
    y = torch.tensor(labels)

    # train / temp
    train_idx, temp_idx = train_test_split(
        range(len(y)), test_size=0.3, stratify=y, random_state=42
    )

    # val / test
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.5, stratify=y[temp_idx], random_state=42
    )

    def make_dataset(idx):
        return TensorDataset(X_ids[idx], X_mask[idx], y[idx])

    return make_dataset(train_idx), make_dataset(val_idx), make_dataset(test_idx)

# ─────────────────────────────────────────────
# TRAIN
# ─────────────────────────────────────────────
def train(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0

    for ids, mask, labels in loader:
        ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)

        optimizer.zero_grad()
        out = model(ids, mask)

        loss = nn.CrossEntropyLoss()(out, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)

# ─────────────────────────────────────────────
# EVALUATE
# ─────────────────────────────────────────────
def evaluate(model, loader, device):
    model.eval()

    preds, labels_all, probs = [], [], []

    with torch.no_grad():
        for ids, mask, labels in loader:
            ids, mask = ids.to(device), mask.to(device)

            out = model(ids, mask)
            prob = torch.softmax(out, dim=1)

            preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            labels_all.extend(labels.numpy())
            probs.extend(prob.cpu().numpy())

    return np.array(preds), np.array(labels_all), np.array(probs)

# ─────────────────────────────────────────────
# METRICS
# ─────────────────────────────────────────────
def print_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted")
    rec = recall_score(y_true, y_pred, average="weighted")
    f1 = f1_score(y_true, y_pred, average="weighted")
    roc = roc_auc_score(y_true, y_prob[:, 1])

    cm = confusion_matrix(y_true, y_pred)
    spec = cm[0, 0] / (cm[0, 0] + cm[0, 1])

    print("\nFINAL TEST PERFORMANCE")
    print("=" * 40)
    print(f"Accuracy    : {acc:.4f}")
    print(f"Precision   : {prec:.4f}")
    print(f"Recall      : {rec:.4f}")
    print(f"Specificity : {spec:.4f}")
    print(f"F1 Score    : {f1:.4f}")
    print(f"ROC-AUC     : {roc:.4f}")
    print("=" * 40)

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

# ─────────────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────────────
def run():
    texts, labels, le = load_mpqa()

    tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

    train_data, val_data, test_data = prepare_data(texts, labels, tokenizer)

    train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=16)
    test_loader = DataLoader(test_data, batch_size=16)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = DeBERTaBiLSTM(num_classes=2).to(device)

    optimizer = AdamW(model.parameters(), lr=2e-5)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=len(train_loader) * 5
    )

    best_val_acc = 0

    for epoch in range(5):
        loss = train(model, train_loader, optimizer, scheduler, device)

        preds, labels_val, _ = evaluate(model, val_loader, device)
        acc = accuracy_score(labels_val, preds)

        print(f"Epoch {epoch+1} | Loss: {loss:.4f} | Val Acc: {acc:.4f}")

        if acc > best_val_acc:
            best_val_acc = acc
            torch.save(model.state_dict(), "best_model.pth")

    print("\nLoading best model...")
    model.load_state_dict(torch.load("best_model.pth"))

    preds, labels_test, probs = evaluate(model, test_loader, device)

    print_metrics(labels_test, preds, probs)

# RUN
if __name__ == "__main__":
    run()

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



Loading MPQA dataset...
label
0    2293
1    2293
Name: count, dtype: int64


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1 | Loss: 0.6783 | Val Acc: 0.6352
Epoch 2 | Loss: 0.5672 | Val Acc: 0.6875
Epoch 3 | Loss: 0.4796 | Val Acc: 0.7369
Epoch 4 | Loss: 0.4188 | Val Acc: 0.8503
Epoch 5 | Loss: 0.3876 | Val Acc: 0.8605

Loading best model...

FINAL TEST PERFORMANCE
Accuracy    : 0.8270
Precision   : 0.8273
Recall      : 0.8270
Specificity : 0.8140
F1 Score    : 0.8270
ROC-AUC     : 0.8943

Confusion Matrix:
[[280  64]
 [ 55 289]]

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.81      0.82       344
           1       0.82      0.84      0.83       344

    accuracy                           0.83       688
   macro avg       0.83      0.83      0.83       688
weighted avg       0.83      0.83      0.83       688



In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, confusion_matrix
)
from sklearn.preprocessing import LabelEncoder
from transformers import AutoModel, AutoTokenizer
from datasets import load_dataset
import re
import warnings
from torch.utils.data import DataLoader, TensorDataset
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

warnings.filterwarnings('ignore')


def clean_text(text):
    text = str(text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)  # remove URLs
    text = re.sub(r'@\w+|#\w+', '', text)                # remove @mentions/#tags
    text = re.sub(r'[^\w\s]', '', text)                  # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

# ─────────────────────────────────────────────
# LOAD DATA
# ─────────────────────────────────────────────
def load_mr():
    print("\nLoading MR (Movie Review) dataset...")
    ds = load_dataset("mattymchen/mr", split="test")
    df = ds.to_pandas()

    df["text"] = df["text"].apply(clean_text)
    df = df[df["text"].str.strip() != ""].reset_index(drop=True)

    le = LabelEncoder()
    df["label"] = le.fit_transform(df["label"])

    # Balance dataset
    min_class = df["label"].value_counts().min()
    df = df.groupby("label").sample(min_class, random_state=42).reset_index(drop=True)

    print(df["label"].value_counts())
    return df["text"].tolist(), df["label"].tolist(), le


class DeBERTaBiLSTM(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.bert = AutoModel.from_pretrained(
            "microsoft/deberta-v3-base",
            torch_dtype=torch.float32
        )

        self.lstm = nn.LSTM(
            input_size=768,
            hidden_size=256,          
            batch_first=True,
            bidirectional=True,
            num_layers=2,            
            dropout=0.2
        )

        # Mean + Max pooling → 256*2 * 2 = 1024-dim input to FC
        self.fc = nn.Sequential(
            nn.Linear(256 * 2 * 2, 256),
            nn.LayerNorm(256),
            nn.GELU(),                
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        x = outputs.last_hidden_state.float()          # (B, T, 768)

        x, _ = self.lstm(x)                            # (B, T, 512)

        mask = attention_mask.unsqueeze(-1).float()    # (B, T, 1)
        x_masked = x * mask

        mean_pool = x_masked.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)  # (B, 512)
        max_pool  = x_masked.max(dim=1).values                              # (B, 512)

        pooled = torch.cat([mean_pool, max_pool], dim=1)   # (B, 1024)
        return self.fc(pooled)

# ─────────────────────────────────────────────
# TOKENIZATION + SPLIT (TRAIN / VAL / TEST)
# ─────────────────────────────────────────────
def prepare_data(texts, labels, tokenizer):
    enc = tokenizer(
        texts, padding=True, truncation=True,
        max_length=128, return_tensors="pt"
    )

    X_ids, X_mask = enc["input_ids"], enc["attention_mask"]
    y = torch.tensor(labels)

    train_idx, temp_idx = train_test_split(
        range(len(y)), test_size=0.3, stratify=y, random_state=42
    )
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.5, stratify=y[temp_idx], random_state=42
    )

    def make_dataset(idx):
        return TensorDataset(X_ids[idx], X_mask[idx], y[idx])

    return make_dataset(train_idx), make_dataset(val_idx), make_dataset(test_idx)

# ─────────────────────────────────────────────
# TRAIN
# ─────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0

    for ids, mask, labels in loader:
        ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)

        optimizer.zero_grad()
        out  = model(ids, mask)
        loss = nn.CrossEntropyLoss()(out, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(loader)

# ─────────────────────────────────────────────
# EVALUATE
# ─────────────────────────────────────────────
def evaluate(model, loader, device):
    model.eval()
    preds, labels_all, probs = [], [], []

    with torch.no_grad():
        for ids, mask, labels in loader:
            ids, mask = ids.to(device), mask.to(device)
            out  = model(ids, mask)
            prob = torch.softmax(out, dim=1)

            preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            labels_all.extend(labels.numpy())
            probs.extend(prob.cpu().numpy())

    return np.array(preds), np.array(labels_all), np.array(probs)

# ─────────────────────────────────────────────
# METRICS
# ─────────────────────────────────────────────
def print_metrics(y_true, y_pred, y_prob):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    roc  = roc_auc_score(y_true, y_prob[:, 1])
    cm   = confusion_matrix(y_true, y_pred)
    spec = cm[0, 0] / (cm[0, 0] + cm[0, 1]) if (cm[0, 0] + cm[0, 1]) > 0 else 0.0

    print("\nFINAL TEST PERFORMANCE")
    print("=" * 40)
    print(f"Accuracy    : {acc:.4f}")
    print(f"Precision   : {prec:.4f}")
    print(f"Recall      : {rec:.4f}")
    print(f"Specificity : {spec:.4f}")
    print(f"F1 Score    : {f1:.4f}")
    print(f"ROC-AUC     : {roc:.4f}")
    print("=" * 40)
    print("\nConfusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

# ─────────────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────────────
def run():
    texts, labels, le = load_mr()

    tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

    train_data, val_data, test_data = prepare_data(texts, labels, tokenizer)

    train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
    val_loader   = DataLoader(val_data,   batch_size=16)
    test_loader  = DataLoader(test_data,  batch_size=16)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nUsing device: {device}")

    model = DeBERTaBiLSTM(num_classes=2).to(device)


    optimizer = AdamW([
        {"params": model.bert.parameters(), "lr": 1e-5},   # slow for BERT
        {"params": model.lstm.parameters(), "lr": 1e-3},   # fast for LSTM
        {"params": model.fc.parameters(),   "lr": 1e-3},   # fast for head
    ], weight_decay=0.01)

    EPOCHS      = 10
    total_steps = len(train_loader) * EPOCHS

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    best_val_acc = 0
    patience     = 3     
    no_improve   = 0

    for epoch in range(EPOCHS):
        loss = train_epoch(model, train_loader, optimizer, scheduler, device)

        preds, labels_val, _ = evaluate(model, val_loader, device)
        acc = accuracy_score(labels_val, preds)

        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss:.4f} | Val Acc: {acc:.4f}")

        if acc > best_val_acc:
            best_val_acc = acc
            no_improve   = 0
            torch.save(model.state_dict(), "best_model_mr.pth")
            print(f"  → Best model saved (Val Acc: {best_val_acc:.4f})")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break

    print("\nLoading best model...")
    model.load_state_dict(torch.load("best_model_mr.pth", map_location=device))

    preds, labels_test, probs = evaluate(model, test_loader, device)
    print_metrics(labels_test, preds, probs)

# RUN
if __name__ == "__main__":
    run()


Loading MR (Movie Review) dataset...
label
0    5331
1    5331
Name: count, dtype: int64

Using device: cuda


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/10 | Loss: 0.6801 | Val Acc: 0.6467
  → Best model saved (Val Acc: 0.6467)
Epoch 2/10 | Loss: 0.6047 | Val Acc: 0.7455
  → Best model saved (Val Acc: 0.7455)
Epoch 3/10 | Loss: 0.5334 | Val Acc: 0.7430
Epoch 4/10 | Loss: 0.5014 | Val Acc: 0.7711
  → Best model saved (Val Acc: 0.7711)
Epoch 5/10 | Loss: 0.4507 | Val Acc: 0.7780
  → Best model saved (Val Acc: 0.7780)
Epoch 6/10 | Loss: 0.4146 | Val Acc: 0.7886
  → Best model saved (Val Acc: 0.7886)
Epoch 7/10 | Loss: 0.3895 | Val Acc: 0.7899
  → Best model saved (Val Acc: 0.7899)
Epoch 8/10 | Loss: 0.3536 | Val Acc: 0.7817
Epoch 9/10 | Loss: 0.3385 | Val Acc: 0.7936
  → Best model saved (Val Acc: 0.7936)
Epoch 10/10 | Loss: 0.3174 | Val Acc: 0.7986
  → Best model saved (Val Acc: 0.7986)

Loading best model...

FINAL TEST PERFORMANCE
Accuracy    : 0.8044
Precision   : 0.8047
Recall      : 0.8044
Specificity : 0.7887
F1 Score    : 0.8043
ROC-AUC     : 0.8864

Confusion Matrix:
[[631 169]
 [144 656]]

Classification Report:
        